# Hospitality Management Analytics - Bronze Data Setup

This notebook sets up the environment and generates bronze layer data for the Hospitality Management Analytics project.

It will:
- Create catalog: `hospitality_project`
- Create schema: `bronze_schema`
- Create volume: `raw` within the schema
- Create folders for each bronze table in the volume
- Generate sample data files with **intentional data quality issues** for students to resolve

**Bronze Tables:**
- `raw_reservations`: Booking records (~8,000 rows)
- `raw_guests`: Guest profiles (~5,000 rows)
- `raw_hotel_inventory`: Physical room details (~2,000 rows)
- `raw_pos_transactions`: Restaurant/Bar/Spa charges (~15,000 rows)
- `raw_housekeeping_logs`: Room cleaning status (~12,000 rows)

## Step 1: Define Environment Variables

In [0]:
# Define catalog, schema, and volume names
CATALOG_NAME = 'hospitality_project'
SCHEMA_NAME = 'bronze_schema'
VOLUME_NAME = 'raw'

# Define the base volume path
VOLUME_PATH = f'/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}'

print(f'Catalog: {CATALOG_NAME}')
print(f'Schema: {SCHEMA_NAME}')
print(f'Volume: {VOLUME_NAME}')
print(f'Volume Path: {VOLUME_PATH}')

## Step 2: Create Catalog

In [0]:
%sql
-- Create catalog if it doesn't exist
CREATE CATALOG IF NOT EXISTS hospitality_project;

## Step 3: Create Schema

In [0]:
%sql
-- Create schema within the catalog
CREATE SCHEMA IF NOT EXISTS hospitality_project.bronze_schema;

## Step 4: Create Volume

In [0]:
%sql
-- Create volume within the schema
CREATE VOLUME IF NOT EXISTS hospitality_project.bronze_schema.raw;

## Step 5: Create Directories in Volume

In [0]:
def create_directory_in_volume(volume_path: str, folder_names: list):
    '''
    Creates multiple directories in the specified volume path using dbutils.fs.

    Parameters:
    - volume_path (str): The base volume path
    - folder_names (list): A list of folder names to create
    '''
    print('----------------------------------------------------------------------------------------')
    for folder in folder_names:
        folder_path = f'{volume_path}/{folder}'
        try:
            # Try to list the directory to check if it exists
            dbutils.fs.ls(folder_path)
            print(f'Directory {folder_path} already exists. No action taken.')
        except:
            # Directory doesn't exist, create it
            dbutils.fs.mkdirs(folder_path)
            print(f'Creating folder: {folder_path}')
    print('----------------------------------------------------------------------------------------\n')

# Create folders for bronze tables
bronze_folders = ['reservations', 'guests', 'hotel_inventory', 'pos_transactions', 'housekeeping_logs']
create_directory_in_volume(VOLUME_PATH, bronze_folders)

## Step 6: Delete Existing Files (if resetting)

In [0]:
def delete_source_files(source_path: str):
    """
    Deletes all files in the specified source volume.

    Parameters:
    - source_path: The path to the volume containing the files to delete
    """
    print(f'\nSearching for files in {source_path} to delete...')
    try:
        files = dbutils.fs.ls(source_path)
        file_list = [f.path for f in files if not f.isDir()]
        if not file_list:
            print(f'No files found in {source_path}.\n')
        else:
            for file_path in file_list:
                print(f'Deleting file: {file_path}')
                dbutils.fs.rm(file_path)
    except Exception as e:
        print(f'Directory {source_path} does not exist or is empty: {e}')

# Delete existing files if resetting
for folder in bronze_folders:
    delete_source_files(f'{VOLUME_PATH}/{folder}/')

## Step 7: Install Faker Library

In [0]:
# Install Faker for generating realistic hospitality data
%pip install faker

## Step 8: Generate Sample Data with Data Quality Issues

This step creates sample JSON files with **intentional data quality issues** that students must resolve:

- **Duplicates**: Some records appear multiple times (especially reservations)
- **Missing Values**: NULL values in critical fields
- **Invalid Formats**: Incorrect date formats, invalid numeric values
- **Data Type Issues**: Numbers stored as strings, dates as strings in wrong format
- **Inconsistent Data**: Mixed case values, inconsistent codes
- **Orphaned Records**: References to non-existent entities
- **Outliers**: Extreme values that may indicate errors
- **Overbooking**: Two reservations assigned to the same room on the same date
- **Late Checkout Logic**: POS transactions occurring after check_out_date
- **Dynamic Pricing**: Weekend prices are 2x higher than weekday prices
- **Loyalty Tier Changes**: Guests upgrade tiers over time (SCD Type 2 challenge)

In [0]:
# dbutils.library.restartPython()

In [0]:

def create_sample_data_with_issues():
    """
    Create sample JSON files with intentional data quality issues using Faker.

    Data includes:
    - raw_guests: ~5,000 records with duplicates, missing values, loyalty tier changes over time
    - raw_hotel_inventory: ~2,000 records with room details
    - raw_reservations: ~8,000 records with overbooking, dynamic pricing, invalid dates
    - raw_pos_transactions: ~15,000 records with late checkout logic, orphaned res_ids
    - raw_housekeeping_logs: ~12,000 records with room status changes
    """
    import json
    import random
    from datetime import datetime, timedelta
    from faker import Faker

    fake = Faker()
    Faker.seed(42)  # For reproducibility
    random.seed(42)

    print("\n----------------Creating sample JSON files with data quality issues----------------")

    # Base date for generating timestamps (last 180 days)
    base_date = datetime(2024, 1, 1, 0, 0, 0)

    # ========== HOTEL INVENTORY DATA (Generate first - needed for reservations) ==========
    print("\n🏨 Generating hotel_inventory data...")
    hotels = [1, 2, 3, 4, 5]  # 5 hotels
    room_types = ['Standard', 'Deluxe', 'Suite', 'Presidential', 'standard', 'DELUXE', 'suite']  # Mixed case
    floors_per_hotel = 10
    rooms_per_floor = 20

    sample_inventory = []
    inventory_map = {}  # Track (hotel_id, room_number) -> room_type for reservations

    for hotel_id in hotels:
        for floor in range(1, floors_per_hotel + 1):
            for room_num in range(1, rooms_per_floor + 1):
                room_number = f"{floor}{room_num:02d}"  # e.g., "101", "205"

                # Issue: Some room types are missing or inconsistent
                if random.random() < 0.08:
                    room_type = None
                else:
                    room_type = random.choice(room_types)
                    if room_type:
                        room_type = room_type.title()  # Normalize for tracking

                # Issue: Some renovation dates are missing or invalid
                if random.random() < 0.1:
                    last_renovated = None
                elif random.random() < 0.1:
                    last_renovated = (base_date - timedelta(days=random.randint(0, 3650))).strftime("%d/%m/%Y")  # Wrong format
                else:
                    last_renovated = (base_date - timedelta(days=random.randint(0, 3650))).strftime("%Y-%m-%d")

                sample_inventory.append({
                    "hotel_id": hotel_id,
                    "room_number": room_number,
                    "room_type": room_type,
                    "floor": floor,
                    "is_smoking": random.choice([True, False, None]),  # Issue: Some missing
                    "last_renovated": last_renovated,
                    "max_occupancy": random.randint(2, 6) if random.random() > 0.05 else None  # Issue: Some missing
                })
                inventory_map[(hotel_id, room_number)] = room_type

    print(f"  Generated {len(sample_inventory)} inventory records")

    # ========== GUESTS DATA ==========
    print("\n👥 Generating guests data...")
    loyalty_tiers = ['Bronze', 'Silver', 'Gold', 'Platinum', 'bronze', 'SILVER', 'gold', None]  # Mixed case and missing
    countries = ['USA', 'UK', 'Canada', 'Germany', 'France', 'Japan', 'Australia', 'India']

    sample_guests = []
    guest_ids_used = set()
    guest_tier_history = {}  # Track tier changes over time for SCD Type 2

    for i in range(5000):
        guest_id = 10000 + i
        name = fake.name()

        # Issue: Some names have inconsistent casing or extra spaces
        if random.random() < 0.1:
            name = name.upper()
        elif random.random() < 0.05:
            name = f"  {name}  "

        # Issue: Some emails are missing or invalid format
        if random.random() < 0.08:
            email = None
        elif random.random() < 0.05:
            email = "invalid-email"  # Invalid format
        else:
            email = fake.email()

        # Issue: Some loyalty tiers are missing or inconsistent
        loyalty_tier = random.choice(loyalty_tiers)
        if loyalty_tier:
            loyalty_tier = loyalty_tier.title()  # Normalize for tracking

        country = random.choice(countries) if random.random() > 0.05 else None  # Issue: Some missing

        # Issue: Some registration dates are missing or invalid format
        if random.random() < 0.08:
            registration_date = None
        elif random.random() < 0.1:
            registration_date = (base_date - timedelta(days=random.randint(0, 1095))).strftime("%d/%m/%Y")  # Wrong format
        else:
            registration_date = (base_date - timedelta(days=random.randint(0, 1095))).strftime("%Y-%m-%d")

        # Issue: Some timestamps are in wrong format
        if random.random() < 0.1:
            updated_at = (base_date + timedelta(days=random.randint(-365, 0))).strftime("%d/%m/%Y %H:%M")  # Wrong format
        elif random.random() < 0.05:
            updated_at = "invalid-timestamp"
        else:
            updated_at = (base_date + timedelta(days=random.randint(-365, 0))).strftime("%Y-%m-%dT%H:%M:%SZ")

        sample_guests.append({
            "guest_id": guest_id,
            "name": name,
            "email": email,
            "loyalty_tier": loyalty_tier,
            "country": country,
            "registration_date": registration_date,
            "updated_at": updated_at
        })
        guest_ids_used.add(guest_id)

        # Track tier history for SCD Type 2 challenge
        if loyalty_tier:
            if guest_id not in guest_tier_history:
                guest_tier_history[guest_id] = []
            guest_tier_history[guest_id].append({
                "tier": loyalty_tier,
                "updated_at": updated_at
            })

    # Issue: Add duplicate guests with email variations
    print("  Adding duplicate guests with email variations...")
    for _ in range(100):
        original_guest = random.choice(sample_guests)
        if original_guest["email"] and original_guest["email"] != "invalid-email":
            dup_guest = original_guest.copy()
            # Email variations: add dots, change domain
            email_parts = dup_guest["email"].split("@")
            if len(email_parts) == 2:
                dup_guest["email"] = email_parts[0] + "+alt@" + email_parts[1]
            dup_guest["guest_id"] = 20000 + len(sample_guests)
            sample_guests.append(dup_guest)
            guest_ids_used.add(dup_guest["guest_id"])

    # Issue: Add exact duplicate guest records
    for _ in range(50):
        dup_guest = random.choice(sample_guests).copy()
        dup_guest["guest_id"] = 20000 + len(sample_guests)
        sample_guests.append(dup_guest)
        guest_ids_used.add(dup_guest["guest_id"])

    # ========== RESERVATIONS DATA ==========
    print("\n📅 Generating reservations data...")
    booking_channels = ['Website', 'Phone', 'Walk-in', 'OTA', 'website', 'PHONE', 'walk-in', None]  # Mixed case
    room_types_list = ['Standard', 'Deluxe', 'Suite', 'Presidential']

    sample_reservations = []
    reservation_ids_used = set()
    reservation_room_map = {}  # Track (hotel_id, room_number, date) -> res_id for overbooking detection
    reservation_checkout_map = {}  # Track check_out dates for late checkout POS

    # Ensure we have valid guest_ids and inventory
    valid_guest_ids = list(guest_ids_used)
    valid_inventory = list(inventory_map.keys())

    if not valid_guest_ids:
        raise ValueError("No guest_ids generated! Cannot create reservations.")
    if not valid_inventory:
        raise ValueError("No inventory generated! Cannot create reservations.")

    overbooking_count = 0

    for i in range(8000):
        res_id = 30000 + i

        # Issue: Some reservations reference non-existent guests
        if random.random() < 0.05:
            guest_id = random.randint(50000, 60000)  # Non-existent guest
        else:
            guest_id = random.choice(valid_guest_ids)

        # Select a valid hotel and room
        hotel_id, room_number = random.choice(valid_inventory)
        room_type = inventory_map[(hotel_id, room_number)]

        # Issue: Some room_numbers are missing or invalid (non-existent rooms)
        if random.random() < 0.05:
            room_number = "9999"  # Non-existent room
        elif random.random() < 0.08:
            room_number = None  # Missing room number

        # Issue: Some room types are missing or inconsistent
        if random.random() < 0.08:
            room_type = None
        elif random.random() < 0.1:
            room_type = random.choice(['standard', 'DELUXE', 'suite'])  # Wrong case

        # Generate check-in and check-out dates
        check_in_date_obj = base_date + timedelta(days=random.randint(0, 150))
        stay_length = random.randint(1, 7)  # 1-7 nights
        check_out_date_obj = check_in_date_obj + timedelta(days=stay_length)

        # Issue: Some dates are missing or invalid format
        if random.random() < 0.08:
            check_in_date = None
        elif random.random() < 0.1:
            check_in_date = check_in_date_obj.strftime("%d/%m/%Y")  # Wrong format
        else:
            check_in_date = check_in_date_obj.strftime("%Y-%m-%d")

        if random.random() < 0.08:
            check_out_date = None
        elif random.random() < 0.1:
            check_out_date = check_out_date_obj.strftime("%m-%d-%Y")  # Wrong format
        else:
            check_out_date = check_out_date_obj.strftime("%Y-%m-%d")
            reservation_checkout_map[res_id] = check_out_date_obj

        # Dynamic Pricing: Weekend prices are 2x higher
        base_price = random.uniform(100, 500)
        is_weekend = check_in_date_obj.weekday() >= 5  # Saturday=5, Sunday=6
        if is_weekend:
            total_price = base_price * 2.0  # Weekend: 2x price
        else:
            total_price = base_price

        # Issue: Some prices are missing, negative, or extreme outliers
        if random.random() < 0.08:
            total_price = None
        elif random.random() < 0.05:
            total_price = round(random.uniform(-100, 0), 2)  # Negative price
        elif random.random() < 0.03:
            total_price = round(random.uniform(10000, 50000), 2)  # Extreme outlier
        else:
            total_price = round(total_price, 2)

        # Issue: Some booking channels are missing or inconsistent
        booking_channel = random.choice(booking_channels)

        # Issue: Some timestamps are in wrong format
        if random.random() < 0.1:
            created_at = (check_in_date_obj - timedelta(days=random.randint(0, 30))).strftime("%d/%m/%Y %H:%M")  # Wrong format
        elif random.random() < 0.05:
            created_at = "invalid-timestamp"
        else:
            created_at = (check_in_date_obj - timedelta(days=random.randint(0, 30))).strftime("%Y-%m-%dT%H:%M:%SZ")

        reservation = {
            "res_id": res_id,
            "guest_id": guest_id,
            "hotel_id": hotel_id,
            "room_number": room_number,  # Add room_number for overbooking detection
            "room_type": room_type,
            "check_in_date": check_in_date,
            "check_out_date": check_out_date,
            "total_price": total_price,
            "booking_channel": booking_channel,
            "created_at": created_at
        }
        sample_reservations.append(reservation)
        reservation_ids_used.add(res_id)

        # Track for overbooking detection
        if check_in_date and check_out_date and check_in_date != "invalid-date" and check_out_date != "invalid-date":
            try:
                check_in = datetime.strptime(check_in_date, "%Y-%m-%d")
                check_out = datetime.strptime(check_out_date, "%Y-%m-%d")
                current_date = check_in
                while current_date < check_out:
                    date_key = (hotel_id, room_number, current_date.strftime("%Y-%m-%d"))
                    if date_key in reservation_room_map:
                        overbooking_count += 1
                        # Intentionally create overbooking
                    reservation_room_map[date_key] = res_id
                    current_date += timedelta(days=1)
            except:
                pass  # Skip invalid dates

    print(f"  Generated {overbooking_count} overbooking scenarios (same room + date)")

    # Issue: Add duplicate reservations
    for _ in range(150):
        dup_res = random.choice(sample_reservations).copy()
        dup_res["res_id"] = 50000 + len(sample_reservations)
        sample_reservations.append(dup_res)
        reservation_ids_used.add(dup_res["res_id"])

    # ========== POS TRANSACTIONS DATA ==========
    print("\n💳 Generating pos_transactions data...")
    categories = ['Food', 'Drink', 'Service', 'Spa', 'food', 'DRINK', 'service', None]  # Mixed case
    food_items = ['Breakfast', 'Lunch', 'Dinner', 'Room Service', 'Snacks']
    drink_items = ['Wine', 'Beer', 'Cocktail', 'Coffee', 'Juice']
    service_items = ['Laundry', 'Concierge', 'Parking', 'WiFi']
    spa_items = ['Massage', 'Facial', 'Spa Package', 'Sauna']

    sample_pos = []
    late_checkout_count = 0

    # Ensure we have valid reservation_ids
    valid_res_ids = list(reservation_ids_used)

    for i in range(15000):
        txn_id = 60000 + i

        # Issue: Some POS transactions reference non-existent reservations (walk-in guests)
        if random.random() < 0.15:  # 15% are walk-ins (NULL res_id)
            res_id = None
            checkout_date = None
        elif random.random() < 0.05:
            res_id = random.randint(70000, 80000)  # Non-existent reservation
            checkout_date = None
        else:
            res_id = random.choice(valid_res_ids)
            checkout_date = reservation_checkout_map.get(res_id)

        # Select category and item
        category = random.choice(categories)
        if category:
            category = category.title()  # Normalize

        if category == 'Food':
            item_name = random.choice(food_items)
        elif category == 'Drink':
            item_name = random.choice(drink_items)
        elif category == 'Service':
            item_name = random.choice(service_items)
        elif category == 'Spa':
            item_name = random.choice(spa_items)
        else:
            item_name = "Misc Item"

        # Issue: Some amounts are missing, negative, or extreme outliers
        if random.random() < 0.08:
            amount = None
        elif random.random() < 0.05:
            amount = round(random.uniform(-50, 0), 2)  # Negative amount
        elif random.random() < 0.03:
            amount = round(random.uniform(10000, 50000), 2)  # Extreme outlier
        else:
            amount = round(random.uniform(10, 200), 2)

        # Late Checkout Logic: Some POS transactions occur after check_out_date
        if checkout_date and random.random() < 0.12:  # 12% are late checkout
            # Transaction happens 1-6 hours after checkout
            txn_timestamp = checkout_date + timedelta(hours=random.randint(1, 6))
            late_checkout_count += 1
        elif res_id and checkout_date:
            # Normal transaction during stay
            txn_timestamp = checkout_date - timedelta(hours=random.randint(1, 72))
        else:
            # Walk-in or no checkout date
            txn_timestamp = base_date + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23))

        # Issue: Some timestamps are in wrong format
        if random.random() < 0.1:
            timestamp = txn_timestamp.strftime("%d/%m/%Y %H:%M")  # Wrong format
        elif random.random() < 0.05:
            timestamp = "invalid-timestamp"
        else:
            timestamp = txn_timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

        # Get guest_id from reservation or generate random
        if res_id and res_id in reservation_ids_used:
            # Find guest_id from reservation
            guest_id = None
            for res in sample_reservations:
                if res["res_id"] == res_id:
                    guest_id = res["guest_id"]
                    break
            if not guest_id:
                guest_id = random.choice(valid_guest_ids) if valid_guest_ids else None
        else:
            guest_id = random.choice(valid_guest_ids) if valid_guest_ids else None

        sample_pos.append({
            "txn_id": txn_id,
            "res_id": res_id,
            "item_name": item_name,
            "category": category,
            "amount": amount,
            "timestamp": timestamp,
            "guest_id": guest_id
        })

    print(f"  Generated {late_checkout_count} late checkout POS transactions (after check_out_date)")

    # Issue: Add duplicate POS transactions
    for _ in range(200):
        dup_pos = random.choice(sample_pos).copy()
        dup_pos["txn_id"] = 110000 + len(sample_pos)
        sample_pos.append(dup_pos)

    # ========== HOUSEKEEPING LOGS DATA ==========
    print("\n🧹 Generating housekeeping_logs data...")
    statuses = ['Clean', 'Dirty', 'Maintenance', 'Out-of-Order', 'clean', 'DIRTY', 'maintenance', None]  # Mixed case
    staff_ids = list(range(50000, 51000))  # 1000 staff members

    sample_housekeeping = []

    # Generate logs for each reservation checkout
    for res in sample_reservations:
        if res["check_out_date"] and res["check_out_date"] != "invalid-date" and res.get("room_number"):
            try:
                checkout_date = datetime.strptime(res["check_out_date"], "%Y-%m-%d")
                hotel_id = res["hotel_id"]
                room_number = res["room_number"]  # Use room_number from reservation

                # Status changes: Dirty -> Clean (after checkout)
                # Issue: Some statuses are missing or inconsistent
                if random.random() < 0.08:
                    status = None
                else:
                    status = random.choice(statuses)
                    if status:
                        status = status.title()  # Normalize

                # Clean status happens 1-4 hours after checkout
                clean_timestamp = checkout_date + timedelta(hours=random.randint(1, 4))

                # Issue: Some timestamps are in wrong format
                if random.random() < 0.1:
                    timestamp = clean_timestamp.strftime("%d/%m/%Y %H:%M")  # Wrong format
                elif random.random() < 0.05:
                    timestamp = "invalid-timestamp"
                else:
                    timestamp = clean_timestamp.strftime("%Y-%m-%dT%H:%M:%SZ")

                sample_housekeeping.append({
                    "log_id": 80000 + len(sample_housekeeping),
                    "hotel_id": hotel_id,
                    "room_number": room_number,
                    "staff_id": random.choice(staff_ids),
                    "status": status,
                    "timestamp": timestamp
                })
            except:
                pass  # Skip invalid dates

    # Add additional housekeeping logs (status changes during stay, etc.)
    for _ in range(12000 - len(sample_housekeeping)):
        hotel_id = random.choice(hotels)
        room_number = random.choice([inv["room_number"] for inv in sample_inventory if inv["hotel_id"] == hotel_id])

        # Issue: Some statuses are missing or inconsistent
        if random.random() < 0.08:
            status = None
        else:
            status = random.choice(statuses)
            if status:
                status = status.title()

        # Issue: Some timestamps are in wrong format
        log_date = base_date + timedelta(days=random.randint(0, 180), hours=random.randint(0, 23))
        if random.random() < 0.1:
            timestamp = log_date.strftime("%d/%m/%Y %H:%M")
        elif random.random() < 0.05:
            timestamp = "invalid-timestamp"
        else:
            timestamp = log_date.strftime("%Y-%m-%dT%H:%M:%SZ")

        sample_housekeeping.append({
            "log_id": 80000 + len(sample_housekeeping),
            "hotel_id": hotel_id,
            "room_number": room_number,
            "staff_id": random.choice(staff_ids),
            "status": status,
            "timestamp": timestamp
        })

    # Issue: Add duplicate housekeeping logs
    for _ in range(150):
        dup_log = random.choice(sample_housekeeping).copy()
        dup_log["log_id"] = 130000 + len(sample_housekeeping)
        sample_housekeeping.append(dup_log)

    # Write files using dbutils.fs.put (newline-delimited JSON)
    try:
        # Guests file
        guests_file = f'{VOLUME_PATH}/guests/00.json'
        guests_json_lines = [json.dumps(guest) for guest in sample_guests]
        guests_content = '\n'.join(guests_json_lines)
        dbutils.fs.put(guests_file, guests_content, overwrite=True)
        print(f'✅ Created guests file: {guests_file} ({len(sample_guests)} records)')

        # Hotel inventory file
        inventory_file = f'{VOLUME_PATH}/hotel_inventory/00.json'
        inventory_json_lines = [json.dumps(inv) for inv in sample_inventory]
        inventory_content = '\n'.join(inventory_json_lines)
        dbutils.fs.put(inventory_file, inventory_content, overwrite=True)
        print(f'✅ Created hotel_inventory file: {inventory_file} ({len(sample_inventory)} records)')

        # Reservations file (split into multiple files)
        reservations_per_file = 3000
        num_res_files = (len(sample_reservations) + reservations_per_file - 1) // reservations_per_file
        for file_num in range(num_res_files):
            start_idx = file_num * reservations_per_file
            end_idx = min(start_idx + reservations_per_file, len(sample_reservations))
            file_reservations = sample_reservations[start_idx:end_idx]
            reservations_file = f'{VOLUME_PATH}/reservations/{file_num:02d}.json'
            reservations_json_lines = [json.dumps(res) for res in file_reservations]
            reservations_content = '\n'.join(reservations_json_lines)
            dbutils.fs.put(reservations_file, reservations_content, overwrite=True)
            print(f'✅ Created reservations file: {reservations_file} ({len(file_reservations)} records)')

        # POS transactions file (split into multiple files)
        pos_per_file = 5000
        num_pos_files = (len(sample_pos) + pos_per_file - 1) // pos_per_file
        for file_num in range(num_pos_files):
            start_idx = file_num * pos_per_file
            end_idx = min(start_idx + pos_per_file, len(sample_pos))
            file_pos = sample_pos[start_idx:end_idx]
            pos_file = f'{VOLUME_PATH}/pos_transactions/{file_num:02d}.json'
            pos_json_lines = [json.dumps(txn) for txn in file_pos]
            pos_content = '\n'.join(pos_json_lines)
            dbutils.fs.put(pos_file, pos_content, overwrite=True)
            print(f'✅ Created pos_transactions file: {pos_file} ({len(file_pos)} records)')

        # Housekeeping logs file (split into multiple files)
        housekeeping_per_file = 4000
        num_housekeeping_files = (len(sample_housekeeping) + housekeeping_per_file - 1) // housekeeping_per_file
        for file_num in range(num_housekeeping_files):
            start_idx = file_num * housekeeping_per_file
            end_idx = min(start_idx + housekeeping_per_file, len(sample_housekeeping))
            file_housekeeping = sample_housekeeping[start_idx:end_idx]
            housekeeping_file = f'{VOLUME_PATH}/housekeeping_logs/{file_num:02d}.json'
            housekeeping_json_lines = [json.dumps(log) for log in file_housekeeping]
            housekeeping_content = '\n'.join(housekeeping_json_lines)
            dbutils.fs.put(housekeeping_file, housekeeping_content, overwrite=True)
            print(f'✅ Created housekeeping_logs file: {housekeeping_file} ({len(file_housekeeping)} records)')

        print(f'\n📊 Total records generated:')
        print(f'  - Guests: {len(sample_guests)} records (guest_id: 10000-14999, plus duplicates)')
        print(f'  - Hotel Inventory: {len(sample_inventory)} records')
        print(f'  - Reservations: {len(sample_reservations)} records (res_id: 30000-37999)')
        print(f'  - POS Transactions: {len(sample_pos)} records (txn_id: 60000+)')
        print(f'  - Housekeeping Logs: {len(sample_housekeeping)} records (log_id: 80000+)')

        # Print relationship summary for verification
        print(f'\n📋 Data Relationship Summary:')
        print(f'  - Guests: {len(sample_guests)} records')
        print(f'    → Includes ~100 duplicate guests with email variations')
        print(f'  - Hotel Inventory: {len(sample_inventory)} records')
        print(f'    → 5 hotels, ~200 rooms per hotel')
        print(f'  - Reservations: {len(sample_reservations)} records')
        print(f'    → References guest_id from guests (95% valid, 5% orphaned)')
        print(f'    → {overbooking_count} overbooking scenarios (same room + date)')
        print(f'    → Weekend prices are 2x higher than weekday prices')
        print(f'  - POS Transactions: {len(sample_pos)} records')
        print(f'    → References res_id from reservations (85% valid, 15% walk-ins, 5% orphaned)')
        print(f'    → {late_checkout_count} late checkout transactions (after check_out_date)')
        print(f'  - Housekeeping Logs: {len(sample_housekeeping)} records')
        print(f'    → References hotel_id and room_number from inventory')
        print(f'\n💡 Join Relationships:')
        print(f'  - reservations → guests (via guest_id)')
        print(f'  - reservations → hotel_inventory (via hotel_id, room_number)')
        print(f'  - pos_transactions → reservations (via res_id)')
        print(f'  - pos_transactions → guests (via guest_id)')
        print(f'  - housekeeping_logs → hotel_inventory (via hotel_id, room_number)')

        return True
    except Exception as e:
        print(f'❌ Error creating sample files: {e}')
        import traceback
        traceback.print_exc()
        return False

# Create sample JSON files with data quality issues
print('\n📝 Creating sample JSON files with intentional data quality issues...')
sample_created = create_sample_data_with_issues()

if sample_created:
    print('\n✅ Successfully created all sample JSON files!')
    print(f'\nFiles created in: {VOLUME_PATH}')
    print('  - guests/00.json (~5,150 guests with data issues)')
    print('  - hotel_inventory/00.json (~2,000 inventory records)')
    print('  - reservations/*.json (~8,150 reservations with overbooking, dynamic pricing)')
    print('  - pos_transactions/*.json (~15,200 POS transactions with late checkout logic)')
    print('  - housekeeping_logs/*.json (~12,150 housekeeping logs with data issues)')
    print('\n⚠️  Note: These files contain intentional data quality issues that you must resolve!')
    print('\n💡 Tip: Use Auto Loader with schema evolution and rescued data column to handle malformed records.')
else:
    print('\n❌ Could not create sample files automatically.')

## Step 9: Verify Setup

In [0]:
%sql
-- Verify catalog exists
SHOW CATALOGS LIKE 'hospitality_project';

In [0]:
%sql
-- Verify schema exists
SHOW SCHEMAS IN hospitality_project LIKE 'bronze_schema';

In [0]:
%sql
-- Verify volume exists
SHOW VOLUMES IN hospitality_project.bronze_schema LIKE 'raw';

In [0]:
# Verify folders and files
print(f'\nVerifying volume structure:')
print(f'Volume path: {VOLUME_PATH}')
print(f'\nFolders:')
for folder in bronze_folders:
    folder_path = f'{VOLUME_PATH}/{folder}'
    try:
        files = dbutils.fs.ls(folder_path)
        file_list = [f.name for f in files if not f.isDir()]
        total_size = sum(f.size for f in files if not f.isDir())
        print(f'  {folder}/: {len(file_list)} file(s), {total_size:,} bytes')
        if file_list:
            for file in sorted(file_list)[:3]:  # Show first 3 files
                print(f'    - {file}')
            if len(file_list) > 3:
                print(f'    ... and {len(file_list) - 3} more file(s)')
    except Exception as e:
        print(f'  {folder}/: NOT FOUND or ERROR - {e}')

## Setup Complete!

Your environment is now configured with:
- Catalog: `hospitality_project`
- Schema: `bronze_schema`
- Volume: `raw` at `/Volumes/hospitality_project/bronze_schema/raw`
- Folders: `reservations`, `guests`, `hotel_inventory`, `pos_transactions`, `housekeeping_logs`

**Bronze Data Summary:**
- `raw_guests`: ~5,150 records (includes duplicate guests with email variations)
- `raw_hotel_inventory`: ~2,000 records (5 hotels, ~200 rooms per hotel)
- `raw_reservations`: ~8,150 records (includes overbooking scenarios, dynamic pricing)
- `raw_pos_transactions`: ~15,200 records (includes late checkout transactions)
- `raw_housekeeping_logs`: ~12,150 records

**⚠️ Important:** The sample data contains intentional data quality issues that you must identify and resolve in your pipeline:
- Missing values (NULL)
- Duplicate records (especially reservations and guests with email variations)
- Invalid date/timestamp formats
- Out-of-range values (negative prices, invalid dates, etc.)
- Inconsistent case (mixed uppercase/lowercase)
- Orphaned records (references to non-existent entities)
- **Overbooking**: Two reservations assigned to the same room on the same date
- **Late checkout logic**: POS transactions occurring after check_out_date
- **Dynamic pricing**: Weekend prices are 2x higher than weekday prices
- **Loyalty tier changes**: Guests upgrade tiers over time (SCD Type 2 challenge)
- Data type issues (numbers as strings, wrong formats)

**💡 Key Techniques to Use:**
- Auto Loader with schema evolution
- Rescued data column for malformed records
- PySpark functions for data cleaning (filter, when, regexp_replace, etc.)
- Guest deduplication logic (same email + name similarity)
- SCD Type 2 implementation for loyalty tier tracking
- Date explosion technique for room availability (date-split exercise)
- Overbooking detection using window functions
- Total folio calculation (reservation + POS aggregation)
- Data quality checks and validations

You can now proceed to the requirements notebook to understand the project objectives.